# Synthetic Data Generation for Inference Simulation

Generates **100,000 fully synthetic sessions** (none from the original dataset) to simulate a real inference flow.

- The same `CorrelatedAugmenter` from Notebook 4 is used to preserve distributions and correlations
- Target distribution: **~1.5% vishing** (98,500 legitimate + 1,500 vishing)
- The original 50K data is used only as a **statistical reference** for the augmenter, it is not included in the output
- Output: `s3://poc-fraude-vishing/proyecto/data/inference_simulation/inference_100k.parquet`

In [ ]:
%pip install --quiet "sagemaker<3"

In [ ]:
import pandas as pd
import numpy as np
import boto3
import sagemaker
import time
import warnings
from datetime import datetime, timedelta
from scipy import stats as scipy_stats

warnings.filterwarnings('ignore')
np.random.seed(42)

sagemaker_session = sagemaker.Session()
role              = sagemaker.get_execution_role()
s3_client         = boto3.client('s3')

BUCKET      = 'poc-fraude-vishing'
BASE_PREFIX = 'proyecto'

S3_SOURCE_PATH = f's3://{BUCKET}/{BASE_PREFIX}/raw_data/biocatch_sinthetic_data.csv'
S3_OUTPUT_PATH = f's3://{BUCKET}/{BASE_PREFIX}/data/inference_simulation/inference_100k.parquet'

print(f'Source      : {S3_SOURCE_PATH}')
print(f'Destination : {S3_OUTPUT_PATH}')

## 1. Loading the original dataset (statistical reference)

In [ ]:
df_original = pd.read_csv(S3_SOURCE_PATH, parse_dates=['session_timestamp'])

print(f'Original dataset : {len(df_original):,} rows × {df_original.shape[1]} columns')
print(f'  Legitimate: {(df_original["is_vishing"]==0).sum():,}')
print(f'  Vishing   : {(df_original["is_vishing"]==1).sum():,}  ({df_original["is_vishing"].mean()*100:.1f}%)')

## 2. Column definitions by type

In [ ]:
continuous_cols = [
    'avg_keyhold_ms', 'avg_interkey_latency_ms', 'typing_speed_cps',
    'keystroke_variability', 'segmented_typing_ratio',
    'avg_touch_pressure', 'avg_touch_size_px', 'swipe_speed_px_s',
    'swipe_directional_variance', 'scroll_speed_avg',
    'device_tilt_angle_mean', 'device_tilt_variability',
    'gyro_rotation_rate_mean', 'accelerometer_jerk_mean',
    'avg_hesitation_duration_s', 'max_hesitation_duration_s',
    'total_dead_time_s', 'dead_time_ratio',
    'screen_transition_time_avg_s', 'data_familiarity_score',
    'session_duration_s', 'call_overlap_duration_s',
    'time_to_transaction_s',
    'errors_per_minute', 'interactions_per_s', 'hesitation_composite',
]

integer_cols = [
    'phone_motion_events', 'hesitation_count', 'dead_time_periods',
    'screens_visited', 'unique_screens_visited', 'unusual_screen_visits',
    'navigation_back_count', 'input_error_count', 'input_correction_count',
    'amount_field_corrections', 'beneficiary_field_corrections',
    'copy_paste_events', 'doodling_events', 'hour_of_day',
    'transaction_amount_cop',
]

binary_cols = [
    'is_atypical_hour', 'phone_call_active',
    'remote_access_tool_detected', 'suspicious_app_detected',
    'transaction_attempted', 'is_new_beneficiary',
]

# Features the model will use (excludes BioCatch scores and other discarded ones)
MODEL_FEATURES = continuous_cols + integer_cols + binary_cols

print(f'Continuous features : {len(continuous_cols)}')
print(f'Integer features    : {len(integer_cols)}')
print(f'Binary features     : {len(binary_cols)}')
print(f'Total for the model : {len(MODEL_FEATURES)}')

## 3. CorrelatedAugmenter

In [ ]:
class CorrelatedAugmenter:
    """
    Generates synthetic data preserving marginal distributions,
    correlation structure, and domain constraints.
    Identical to the one used in Notebook 4.
    """

    def __init__(self, df_class, continuous_cols, integer_cols, binary_cols):
        self.continuous_cols = continuous_cols
        self.integer_cols    = integer_cols
        self.binary_cols     = binary_cols
        self.all_cols        = continuous_cols + integer_cols + binary_cols

        self.df_class  = df_class[self.all_cols].copy()
        self.means     = self.df_class.mean()
        self.stds      = self.df_class.std()
        self.mins      = self.df_class.min()
        self.maxs      = self.df_class.max()

        self.binary_probs = {col: df_class[col].mean() for col in binary_cols}

        self.integer_distributions = {}
        for col in integer_cols:
            vals          = df_class[col].values
            unique, counts = np.unique(vals, return_counts=True)
            self.integer_distributions[col] = (unique, counts / counts.sum())

        cont_data          = df_class[continuous_cols].values
        self.corr_matrix   = np.corrcoef(cont_data.T) + np.eye(len(continuous_cols)) * 1e-6
        self.cholesky      = np.linalg.cholesky(self.corr_matrix)

        self.percentiles = {
            col: np.sort(cont_data[:, i])
            for i, col in enumerate(continuous_cols)
        }

    def generate(self, n_samples, noise_scale=0.03):
        result = pd.DataFrame(index=range(n_samples))

        z            = np.random.normal(0, 1, (n_samples, len(self.continuous_cols)))
        z_correlated = z @ self.cholesky.T
        u            = scipy_stats.norm.cdf(z_correlated)

        for i, col in enumerate(self.continuous_cols):
            sorted_vals = self.percentiles[col]
            n_orig      = len(sorted_vals)
            indices     = np.clip((u[:, i] * n_orig).astype(int), 0, n_orig - 1)
            values      = sorted_vals[indices]
            values      = values + np.random.normal(0, self.stds[col] * noise_scale, n_samples)
            result[col] = values

        for col in self.integer_cols:
            unique, probs = self.integer_distributions[col]
            sampled       = np.random.choice(unique, size=n_samples, p=probs)
            perturbation  = np.random.choice([-1, 0, 0, 0, 1], size=n_samples)
            mask          = np.random.random(n_samples) < noise_scale * 5
            result[col]   = sampled + perturbation * mask

        for col in self.binary_cols:
            result[col] = (np.random.random(n_samples) < self.binary_probs[col]).astype(int)

        self._enforce_constraints(result)
        return result

    def _enforce_constraints(self, df):
        for col in ['segmented_typing_ratio', 'avg_touch_pressure', 'dead_time_ratio',
                    'data_familiarity_score', 'keystroke_variability', 'swipe_directional_variance']:
            if col in df.columns:
                df[col] = df[col].clip(0, 1)

        for col in self.integer_cols:
            if col in df.columns:
                df[col] = df[col].round().clip(lower=0).astype(int)

        for col in self.binary_cols:
            if col in df.columns:
                df[col] = df[col].round().clip(0, 1).astype(int)

        if 'hour_of_day' in df.columns:
            df['hour_of_day'] = df['hour_of_day'].clip(0, 23)

        if 'device_tilt_angle_mean' in df.columns:
            df['device_tilt_angle_mean'] = df['device_tilt_angle_mean'].clip(0, 90)

        for col in self.continuous_cols:
            if col in df.columns and self.mins[col] >= 0:
                df[col] = df[col].clip(lower=0)

        if 'unique_screens_visited' in df.columns and 'screens_visited' in df.columns:
            df['unique_screens_visited'] = np.minimum(
                df['unique_screens_visited'], df['screens_visited'])

        if 'call_overlap_duration_s' in df.columns and 'phone_call_active' in df.columns:
            df.loc[df['phone_call_active'] == 0, 'call_overlap_duration_s'] = 0

        if 'time_to_transaction_s' in df.columns and 'transaction_attempted' in df.columns:
            df.loc[df['transaction_attempted'] == 0, 'time_to_transaction_s'] = 0

        if 'transaction_amount_cop' in df.columns and 'transaction_attempted' in df.columns:
            df.loc[df['transaction_attempted'] == 0, 'transaction_amount_cop'] = 0

        if 'is_new_beneficiary' in df.columns and 'transaction_attempted' in df.columns:
            df.loc[df['transaction_attempted'] == 0, 'is_new_beneficiary'] = 0


print('CorrelatedAugmenter defined.')

## 4. Generation of 100K synthetic observations

In [ ]:
N_TOTAL        = 100_000
VISHING_PCT    = 0.015
N_VISHING      = int(N_TOTAL * VISHING_PCT)   # 1,500
N_LEGIT        = N_TOTAL - N_VISHING           # 98,500

print(f'Target: {N_TOTAL:,} fully synthetic sessions')
print(f'  Legitimate : {N_LEGIT:,}  ({N_LEGIT/N_TOTAL*100:.1f}%)')
print(f'  Vishing    : {N_VISHING:,}   ({N_VISHING/N_TOTAL*100:.1f}%)')

df_orig_legit   = df_original[df_original['is_vishing'] == 0]
df_orig_vishing = df_original[df_original['is_vishing'] == 1]

print('\nTraining augmenters...')
t0 = time.time()
aug_legit   = CorrelatedAugmenter(df_orig_legit,   continuous_cols, integer_cols, binary_cols)
aug_vishing = CorrelatedAugmenter(df_orig_vishing, continuous_cols, integer_cols, binary_cols)
print(f'  Augmenters ready in {time.time()-t0:.1f}s')

print('\nGenerating legitimate sessions...')
t0 = time.time()
df_synth_legit = aug_legit.generate(N_LEGIT, noise_scale=0.03)
df_synth_legit['is_vishing'] = 0
print(f'  {len(df_synth_legit):,} rows in {time.time()-t0:.1f}s')

print('\nGenerating vishing sessions...')
t0 = time.time()
df_synth_vishing = aug_vishing.generate(N_VISHING, noise_scale=0.03)
df_synth_vishing['is_vishing'] = 1
print(f'  {len(df_synth_vishing):,} rows in {time.time()-t0:.1f}s')

In [ ]:
# Assemble and shuffle
df_inference = pd.concat([df_synth_legit, df_synth_vishing], ignore_index=True)
df_inference = df_inference.sample(frac=1, random_state=42).reset_index(drop=True)

# Add session_id and timestamp to simulate real sessions
df_inference.insert(0, 'session_id',
    [f'INF-{i:07d}' for i in range(1, len(df_inference) + 1)])

base_date  = datetime(2025, 6, 1)
timestamps = [
    base_date + timedelta(
        days=np.random.randint(0, 180),
        hours=int(df_inference.loc[i, 'hour_of_day']),
        minutes=np.random.randint(0, 60),
        seconds=np.random.randint(0, 60),
    )
    for i in range(len(df_inference))
]
df_inference.insert(1, 'session_timestamp', timestamps)

n  = len(df_inference)
nv = int(df_inference['is_vishing'].sum())
print(f'Inference dataset assembled')
print(f'  Total      : {n:,}')
print(f'  Legitimate : {n-nv:,}  ({(n-nv)/n*100:.2f}%)')
print(f'  Vishing    : {nv:,}   ({nv/n*100:.2f}%)')
print(f'  Columns    : {df_inference.shape[1]}')

## 5. Quick distribution validation

In [ ]:
import matplotlib.pyplot as plt

key_feats = [
    'typing_speed_cps', 'hesitation_count', 'segmented_typing_ratio',
    'data_familiarity_score', 'input_correction_count', 'transaction_amount_cop',
]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for ax, feat in zip(axes.flat, key_feats):
    orig_l = df_orig_legit[feat]
    orig_v = df_orig_vishing[feat]
    inf_l  = df_inference[df_inference['is_vishing'] == 0][feat]
    inf_v  = df_inference[df_inference['is_vishing'] == 1][feat]

    ax.hist(orig_l.sample(min(2000, len(orig_l))), bins=30, alpha=0.4,
            color='green', label='Original legit', density=True)
    ax.hist(inf_l.sample(min(2000, len(inf_l))),   bins=30, alpha=0.4,
            color='blue',  label='Synth legit',    density=True)
    ax.hist(inf_v.values,                           bins=20, alpha=0.5,
            color='red',   label='Synth vishing',  density=True)
    ax.set_title(feat, fontsize=10)
    ax.legend(fontsize=7)

plt.suptitle('Distributions: Original vs Synthetic Inference', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Save to S3

In [ ]:
print(f'Saving to {S3_OUTPUT_PATH}...')
t0 = time.time()

df_inference.to_parquet(S3_OUTPUT_PATH, engine='pyarrow', index=False)

print(f'Saved in {time.time()-t0:.1f}s')
print(f'\nInference dataset available at:')
print(f'  {S3_OUTPUT_PATH}')
print(f'\nUse this path in the inference simulator (10_Realtime_Simulation.ipynb)')